In [1]:
import pandas as pd
import numpy as np
import ipaddress
from pathlib import Path

OUT_DIR = Path("/home/chupchik/Рабочий стол/fisrt_stepD/output")
parquet_path = OUT_DIR / "ait_ads_wazuh.parquet"

df = pd.read_parquet(parquet_path)
print("Loaded:", df.shape)
df.head(3)

Loaded: (2600263, 14)


,file,timestamp,location,agent_id,agent_name,agent_ip,hostname,program,full_log,rule_id,rule_description,rule_groups,rule_level,rule_groups_str
0,fox_wazuh.json,2022-01-15 02:32:32+00:00,/var/log/syslog,18,wazuh-client,172.17.131.81,mail,freshclam,Jan 15 02:32:32 mail freshclam[29266]: Sat Jan...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"
1,fox_wazuh.json,2022-01-15 02:32:32+00:00,/var/log/syslog,6,wazuh-client,192.168.128.170,taylorcruz-mail,freshclam,Jan 15 02:32:32 taylorcruz-mail freshclam[2851...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"
2,fox_wazuh.json,2022-01-15 02:32:37+00:00,/var/log/syslog,18,wazuh-client,172.17.131.81,mail,freshclam,Jan 15 02:32:37 mail freshclam[29266]: Sat Jan...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"


In [4]:
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
df = df.dropna(subset=["timestamp"]).copy()

def map_priority(level):
    if pd.isna(level):
        return None
    level = float(level)
    if level <= 3: return "low"
    if level <= 6: return "medium"
    if level <= 10: return "high"
    return "critical"

df["y_priority"] = df["rule_level"].apply(map_priority)
df = df.dropna(subset=["y_priority"]).copy()

df["y_priority"].value_counts()

y_priority
medium      1873973
low          592722
high         131901
critical       1667
Name: count, dtype: int64

In [ ]:
# базовые временные признаки (это базовая подготовка)
df["hour"] = df["timestamp"].dt.hour.astype("int16")
df["dow"] = df["timestamp"].dt.dayofweek.astype("int16")

# базовая информация из agent_ip (internal/external) — оставим как "контекст"
df["agent_ip"] = df["agent_ip"].astype(str)

def is_internal_ip(ip):
    try:
        return int(ipaddress.ip_address(ip).is_private)
    except:
        return 0

df["is_internal_ip"] = df["agent_ip"].apply(is_internal_ip).astype("int8")

# rule_id как базовый структурный признак (это поле правила)
df["rule_id"] = pd.to_numeric(df["rule_id"], errors="coerce").fillna(-1).astype("int32")

df[["hour","dow","is_internal_ip","rule_id"]].head()

,hour,dow,is_internal_ip,rule_id
0,2,5,1,52507
1,2,5,1,52507
2,2,5,1,52507
3,2,5,1,52507
4,2,5,1,52507


In [6]:
y = df["y_priority"].astype(str)

# базовые категориальные
cat_cols = ["location", "program", "agent_name", "hostname"]
cat_cols = [c for c in cat_cols if c in df.columns]

# базовые числовые
num_cols = ["hour", "dow", "is_internal_ip", "rule_id"]

X = df[cat_cols + num_cols].copy()

for c in cat_cols:
    X[c] = X[c].fillna("NA").astype(str)
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0)

print("X:", X.shape, "y:", y.shape)
print("y dist:\n", y.value_counts())
print("cat_cols:", cat_cols)
print("num_cols:", num_cols)

X: (2600263, 8) y: (2600263,)
y dist:
 y_priority
medium      1873973
low          592722
high         131901
critical       1667
Name: count, dtype: int64
cat_cols: ['location', 'program', 'agent_name', 'hostname']
num_cols: ['hour', 'dow', 'is_internal_ip', 'rule_id']


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train dist:\n", y_train.value_counts(normalize=True))

Train: (2080210, 8) Test: (520053, 8)
Train dist:
 y_priority
medium      0.720686
low         0.227947
high        0.050726
critical    0.000641
Name: proportion, dtype: float64


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

# sparse_output=False => dense матрица 
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10, sparse_output=False), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop",
)

gb = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=400,
    max_depth=None,
    min_samples_leaf=30,
    l2_regularization=1e-6,
    random_state=42
)

model_base = Pipeline(steps=[
    ("prep", preprocess),
    ("gb", gb)
])

sw = compute_sample_weight(class_weight="balanced", y=y_train)

model_base.fit(X_train, y_train, gb__sample_weight=sw)
print("Trained baseline ✅")

Trained baseline ✅


In [10]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import pandas as pd

pred = model_base.predict(X_test)

print(classification_report(y_test, pred, digits=3))
print("macro-F1:", f1_score(y_test, pred, average="macro"))

labels = ["low", "medium", "high", "critical"]
cm = confusion_matrix(y_test, pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
cm_df

              precision    recall  f1-score   support

    critical      1.000     1.000     1.000       333
        high      1.000     1.000     1.000     26380
         low      1.000     1.000     1.000    118545
      medium      1.000     1.000     1.000    374795

    accuracy                          1.000    520053
   macro avg      1.000     1.000     1.000    520053
weighted avg      1.000     1.000     1.000    520053

macro-F1: 0.9999986120364514


,pred_low,pred_medium,pred_high,pred_critical
true_low,118545,0,0,0
true_medium,1,374794,0,0
true_high,0,0,26380,0
true_critical,0,0,0,333


In [11]:
err = X_test.copy()
err["y_true"] = y_test.values
err["y_pred"] = pred

pred_crit = (err["y_pred"] == "critical").sum()
true_crit = (err["y_true"] == "critical").sum()
tp_crit = ((err["y_true"] == "critical") & (err["y_pred"] == "critical")).sum()
hc = ((err["y_true"] == "high") & (err["y_pred"] == "critical")).sum()

print("BASELINE")
print("true_critical:", true_crit)
print("predicted_critical:", pred_crit)
print("TP_critical:", tp_crit)
print("high -> critical errors:", hc)

BASELINE
true_critical: 333
predicted_critical: 333
TP_critical: 333
high -> critical errors: 0


In [12]:
# какие rule_id чаще всего у critical
crit_rules = (df[df["y_priority"]=="critical"]["rule_id"]
              .value_counts()
              .head(20))
crit_rules

rule_id
20162    1465
20161     202
Name: count, dtype: int64

In [13]:
# сколько разных классов y_priority встречается в каждом rule_id
tmp = (df.groupby("rule_id")["y_priority"]
         .nunique()
         .value_counts()
         .sort_index())
tmp

y_priority
1    31
Name: count, dtype: int64